In [0]:
%sql
CREATE DATABASE IF NOT EXISTS workspace.gold
COMMENT 'Capa Gold procesados'

In [0]:
spark.sql("DROP TABLE IF EXISTS workspace.gold.fact_predios")

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.gold.fact_predios (
    pk STRING,
    matricula STRING,
    fecha_radica_id INT,
    ubicacion_id INT,
    orip INT,
    divipola INT,
    tipo_predio_id INT,
    categoria_ruralidad_id INT,
    natujur_id INT,
    num_anotacion INT,
    dinamica_2024 INT,
    documento_justificativo STRING,
    count_a INT,
    count_de INT,
    predios_nuevos INT,
    tiene_valor BOOLEAN,
    tiene_mas_de_un_valor BOOLEAN,
    estado_folio STRING,
    valor DOUBLE
)
USING DELTA
""")

In [0]:
import pandas as pd
from pyspark.sql import functions as F

# 1. Leer la tabla Silver a Pandas (Capa de transformación)
# Extraemos los datos de la tabla que acabas de crear en SQL
fact = spark.table("workspace.silver.predios_registro").toPandas()

dim_tipo = spark.table("workspace.gold.dim_tipo_predio").toPandas()
dim_rural = spark.table("workspace.gold.dim_categoria_ruralidad").toPandas()
dim_natujur = spark.table("workspace.gold.dim_natujur").toPandas()
dim_ubicacion = spark.table("workspace.gold.dim_ubicacion").toPandas()


dim_tipo = dim_tipo.drop_duplicates(subset=["tipo_predio_descripcion"])
dim_rural = dim_rural.drop_duplicates(subset=["categoria_ruralidad_descripcion"])
dim_ubicacion = dim_ubicacion.drop_duplicates(subset=["departamento_descripcion", "municipio_descripcion"])


# 2. JOIN con dimensiones

# tipo_predio
fact = fact.merge(
    dim_tipo,
    on="tipo_predio_descripcion",
    how="left"
)

#ruralidad
fact = fact.merge(
    dim_rural,
    on="categoria_ruralidad_descripcion",
    how="left"
)

# ubicacion
fact = fact.drop(columns=["divipola"], errors="ignore")

fact = fact.merge(
    dim_ubicacion,
    on=["departamento_descripcion", "municipio_descripcion"],
    how="left"
)

#- eliminar descripciones
fact = fact.drop(columns=[
    "departamento_descripcion",
    "municipio_descripcion",
    "tipo_predio_descripcion",
    "categoria_ruralidad_descripcion"
], errors="ignore")

# Manejo de nulos (calidad de datos)
fact["estado_folio"] = fact["estado_folio"].fillna("NO INFORMADO")
fact["valor"] = fact["valor"].fillna(0)

# 3. Procesamiento de Fechas
# Convertimos 'fecha_radica' a formato datetime de Pandas para extraer la llave
dates = pd.to_datetime(fact["fecha_radica"], errors="coerce")

# 4. Crear columnas derivadas y limpieza final
fact["fecha_radica_id"] = dates.dt.strftime("%Y%m%d")

fact["fecha_radica_id"] = fact["fecha_radica_id"].fillna("0")
fact["fecha_radica_id"] = fact["fecha_radica_id"].astype(int)



# columnas finales correctas
fact = fact[[
    "pk",
    "matricula",
    "fecha_radica_id",
    "orip",
    "divipola",
    "ubicacion_id",
    "tipo_predio_id",
    "categoria_ruralidad_id",
    "natujur_id",
    "num_anotacion",
    "dinamica_2024",
    "documento_justificativo",
    "count_a",
    "count_de",
    "predios_nuevos",
    "tiene_valor",
    "tiene_mas_de_un_valor",
    "estado_folio",
    "valor"
]]

    # Calidad de datos: Eliminamos filas sin PK (Primary Key) y duplicados
fact = fact.dropna(subset=["pk"])
fact = fact.drop_duplicates(subset=["pk"])


# 5. Convertir de regreso a Spark para guardar en Delta
# Esto devuelve los datos al entorno distribuido de Databricks
df_spark = spark.createDataFrame(fact)


In [0]:
df_spark = df_spark.withColumn(
    "fecha_radica_id",
    F.col("fecha_radica_id").cast("int")
)

In [0]:
from pyspark.sql import functions as F

# convertir columnas
df_spark = df_spark.withColumn(
    "fecha_radica_id",
    F.col("fecha_radica_id").cast("int")
).withColumn(
    "categoria_ruralidad_id",
    F.col("categoria_ruralidad_id").cast("int")
)

# Usamos overwrite para reemplazar completamente los datos existentes
df_spark.write.format("delta") \
    .option("mergeSchema", "true") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold.fact_predios")

In [0]:
print([col for col in fact.columns if "natujur" in col])